# Módulo 03 · Actividad individual: Limpieza, integración y transformación

> *Garbage in, garbage out*: si los datos entran mal, el mejor modelo del mundo saldrá mal.

**Objetivos**
- Comprender por qué preparar los datos es (casi siempre) la fase que más tiempo toma.
- Practicar las 3 tareas: **limpiar**, **integrar** y **transformar**.
- Tomar decisiones **con criterio**, no "porque sí".

> Basado en `sesion_3_preparacion_de_datos.ipynb` (profesor Oscar N. Gómez).

```
  1. LIMPIEZA            2. INTEGRACIÓN           3. TRANSFORMACIÓN
  nulos, duplicados,     unir tablas (joins)      escalar, codificar,
  formatos, errores                               crear variables
```

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler

---
## 1. Limpieza de datos

| Problema | Opciones |
|---|---|
| **Valores perdidos** | Imputar (media, mediana, moda, un valor fijo), eliminar el registro, o usar un modelo para estimarlos |
| **Duplicados** | Eliminarlos para que cada registro sea único |
| **Formatos inconsistentes** | Estandarizar fechas, mayúsculas y minúsculas, unidades y monedas |

### 1.1 Un dataset pequeño con problemas

In [2]:
data = pd.DataFrame(
    {
        "id": [1, 2, 3, 4, 5],
        "cliente": ["Juan P.", "Maria G", "Carlos S.", "Juan P.", None],
        "edad": [28, 34, None, 28, 22],
        "precio": [100.5, None, 85.3, 100.5, 50.0],
        "fecha": ["2023/12/01", "2023/12/02", "2023/12/01", "2023/12/01", "2023/12/03"],
    }
)
data

,id,cliente,edad,precio,fecha
0,1,Juan P.,28.0,100.5,2023/12/01
1,2,Maria G,34.0,NaN,2023/12/02
2,3,Carlos S.,NaN,85.3,2023/12/01
3,4,Juan P.,28.0,100.5,2023/12/01
4,5,NaN,22.0,50.0,2023/12/03


**Antes de tocar nada, diagnosticamos:**

In [3]:
print("Nulos por columna:\n", data.isna().sum(), sep="")
print("\nTipos:\n", data.dtypes, sep="")
print("\n¿Duplicados (ignorando id)?", data.drop(columns="id").duplicated().sum())

Nulos por columna:
id         0
cliente    1
edad       1
precio     1
fecha      0
dtype: int64

Tipos:
id           int64
cliente        str
edad       float64
precio     float64
fecha          str
dtype: object

¿Duplicados (ignorando id)? 1


Observa: la fila 4 es **igual** a la 1 salvo el `id`. `drop_duplicates()` sobre todo el DataFrame **no** la detectaría, porque el `id` es distinto.
Por eso hay que pensar **qué columnas definen un duplicado**.

### 1.2 Duplicados: primero, antes de imputar

¿Por qué primero? Si imputamos con la media **antes** de quitar duplicados, los duplicados "pesan doble" en esa media.

In [4]:
data = data.drop_duplicates(subset=["cliente", "edad", "precio", "fecha"]).reset_index(drop=True)
data

,id,cliente,edad,precio,fecha
0,1,Juan P.,28.0,100.5,2023/12/01
1,2,Maria G,34.0,NaN,2023/12/02
2,3,Carlos S.,NaN,85.3,2023/12/01
3,5,NaN,22.0,50.0,2023/12/03


### 1.3 Imputar valores perdidos

**Opción A: con pandas.**

In [5]:
data_a = data.copy()
data_a["edad"] = data_a["edad"].fillna(data_a["edad"].mean())
data_a["precio"] = data_a["precio"].fillna(data_a["precio"].median())
data_a

,id,cliente,edad,precio,fecha
0,1,Juan P.,28.0,100.5,2023/12/01
1,2,Maria G,34.0,85.3,2023/12/02
2,3,Carlos S.,28.0,85.3,2023/12/01
3,5,NaN,22.0,50.0,2023/12/03


**Opción B: con `SimpleImputer` de scikit-learn.** Hace lo mismo, pero **recuerda** el valor aprendido (`statistics_`) para aplicarlo después a datos nuevos.
Esto será clave en el módulo 05 (pipelines).

In [6]:
imputador = SimpleImputer(strategy="median")
data_b = data.copy()
data_b[["edad", "precio"]] = imputador.fit_transform(data_b[["edad", "precio"]])
print("Valores aprendidos (mediana):", imputador.statistics_)
data_b

Valores aprendidos (mediana): [28.  85.3]


,id,cliente,edad,precio,fecha
0,1,Juan P.,28.0,100.5,2023/12/01
1,2,Maria G,34.0,85.3,2023/12/02
2,3,Carlos S.,28.0,85.3,2023/12/01
3,5,NaN,22.0,50.0,2023/12/03


#### ¿Tiene sentido imputar con la media?

| Estrategia | Cuándo sirve | Riesgo |
|---|---|---|
| **Media** | Distribución simétrica, pocos nulos | Sensible a *outliers* |
| **Mediana** | Distribución asimétrica (precios, ingresos) | Ignora relaciones con otras variables |
| **Moda** | Categóricas | Infla la categoría más común |
| **Valor fijo** ("Desconocido", 0, -1) | Cuando el nulo **significa algo** ("no tiene") | Un 0 falso puede confundir al modelo |
| **Por grupo** | Cuando la variable depende de otra (p.ej. el precio según la ciudad) | Requiere grupos con datos suficientes |
| **Eliminar la fila** | Muy pocos nulos y al azar | Pierdes datos; puede sesgar la muestra |

> **Regla práctica:** antes de imputar, pregúntate *¿por qué falta este dato?* Si la respuesta es "porque no existe" (no tiene piscina), no se imputa: se codifica.

### 1.4 Formatos: fechas y texto

In [7]:
data["fecha"] = pd.to_datetime(data["fecha"], format="%Y/%m/%d", errors="coerce")
data["cliente"] = data["cliente"].fillna("Sin nombre")
data.dtypes

id                  int64
cliente               str
edad              float64
precio            float64
fecha      datetime64[us]
dtype: object

`errors="coerce"` convierte en `NaT` (nulo de fecha) lo que no se pueda leer, en lugar de lanzar un error. **Después hay que revisar cuántos `NaT` quedaron.**

---
## 2. Integración de datos (joins)

Los datos casi nunca vienen en una sola tabla. Hay que **combinarlos** con una **llave**.

In [8]:
clientes = pd.DataFrame(
    {"customer_id": [1, 2, 3, 4], "name": ["Juan P.", "Maria G.", "Carlos S.", "Fernando M."], "age": [28, 34, 29, 20]}
)
ventas = pd.DataFrame(
    {
        "customer_id": [1, 3, 4, 5],
        "purchase_amount": [100.5, 85.3, 45.0, 35.6],
        "purchase_date": ["2023-12-01", "2023-12-02", "2023-12-03", "2023-12-03"],
    }
)
paises = pd.DataFrame({"user_id": [1, 2, 3, 4], "country": ["US", "UK", "MX", "AR"]})
display(clientes, ventas)

,customer_id,name,age
0,1,Juan P.,28
1,2,Maria G.,34
2,3,Carlos S.,29
3,4,Fernando M.,20


,customer_id,purchase_amount,purchase_date
0,1,100.5,2023-12-01
1,3,85.3,2023-12-02
2,4,45.0,2023-12-03
3,5,35.6,2023-12-03


### Tipos de join

```
  INNER                LEFT                 RIGHT                OUTER
  ┌───┬───┬───┐        ┌───┬───┬───┐        ┌───┬───┬───┐        ┌───┬───┬───┐
  │   │███│   │        │███│███│   │        │   │███│███│        │███│███│███│
  └───┴───┴───┘        └───┴───┴───┘        └───┴───┴───┘        └───┴───┴───┘
  solo lo común        todo lo izquierdo    todo lo derecho      todo
```

Observa: el cliente **2** (María) no tiene ventas, y la venta del cliente **5** no tiene cliente registrado.

In [9]:
for como in ["inner", "left", "right", "outer"]:
    resultado = pd.merge(clientes, ventas, on="customer_id", how=como)
    print(f"--- how='{como}' → {len(resultado)} filas")
    print(resultado[["customer_id", "name", "purchase_amount"]].to_string(index=False), "\n")

--- how='inner' → 3 filas
 customer_id        name  purchase_amount
           1     Juan P.            100.5
           3   Carlos S.             85.3
           4 Fernando M.             45.0 

--- how='left' → 4 filas
 customer_id        name  purchase_amount
           1     Juan P.            100.5
           2    Maria G.              NaN
           3   Carlos S.             85.3
           4 Fernando M.             45.0 

--- how='right' → 4 filas
 customer_id        name  purchase_amount
           1     Juan P.            100.5
           3   Carlos S.             85.3
           4 Fernando M.             45.0
           5         NaN             35.6 

--- how='outer' → 5 filas
 customer_id        name  purchase_amount
           1     Juan P.            100.5
           2    Maria G.              NaN
           3   Carlos S.             85.3
           4 Fernando M.             45.0
           5         NaN             35.6 



**`indicator=True`** te dice de dónde vino cada fila. Es muy útil para **auditar** un join:

In [10]:
pd.merge(clientes, ventas, on="customer_id", how="outer", indicator=True)[["customer_id", "name", "purchase_amount", "_merge"]]

,customer_id,name,purchase_amount,_merge
0,1,Juan P.,100.5,both
1,2,Maria G.,NaN,left_only
2,3,Carlos S.,85.3,both
3,4,Fernando M.,45.0,both
4,5,NaN,35.6,right_only


### Llaves con nombres distintos, sufijos y llaves múltiples

In [11]:
# Llaves con nombre distinto
pd.merge(clientes, paises, left_on="customer_id", right_on="user_id", how="inner")

,customer_id,name,age,user_id,country
0,1,Juan P.,28,1,US
1,2,Maria G.,34,2,UK
2,3,Carlos S.,29,3,MX
3,4,Fernando M.,20,4,AR


In [12]:
# Columnas repetidas → sufijos
nuevos_nombres = pd.DataFrame({"customer_id": [1, 2, 3, 4], "name": ["Beto", "Ana", "Carlos", "Daniela"]})
pd.merge(clientes, nuevos_nombres, on="customer_id", suffixes=("_old", "_new"))

,customer_id,name_old,age,name_new
0,1,Juan P.,28,Beto
1,2,Maria G.,34,Ana
2,3,Carlos S.,29,Carlos
3,4,Fernando M.,20,Daniela


In [13]:
# Llave compuesta: ciudad + año
ventas_ciudad = pd.DataFrame(
    {"city": ["NY", "NY", "LA", "LA", "TX"], "year": [2020, 2021, 2020, 2021, 2020], "sales": [100, 150, 200, 180, 220]}
)
ganancias_ciudad = pd.DataFrame(
    {"city": ["NY", "NY", "LA", "TX", "TX"], "year": [2020, 2022, 2020, 2020, 2021], "profit": [30, 70, 50, 40, 60]}
)
pd.merge(ventas_ciudad, ganancias_ciudad, on=["city", "year"], how="outer")

,city,year,sales,profit
0,LA,2020,200.0,50.0
1,LA,2021,180.0,NaN
2,NY,2020,100.0,30.0
3,NY,2021,150.0,NaN
4,NY,2022,NaN,70.0
5,TX,2020,220.0,40.0
6,TX,2021,NaN,60.0


**La trampa de los joins: las filas se multiplican.** Si la llave se repite en las dos tablas, cada combinación genera una fila.
Usa `validate=` para que pandas te avise:

In [14]:
# Juan (customer_id=1) hace una segunda compra
ventas_2 = pd.concat([ventas, pd.DataFrame({"customer_id": [1], "purchase_amount": [20.0], "purchase_date": ["2023-12-05"]})])

print("Filas de clientes:", len(clientes), "· filas del join:", len(pd.merge(clientes, ventas_2, on="customer_id")))

try:
    pd.merge(clientes, ventas_2, on="customer_id", validate="one_to_one")
except pd.errors.MergeError as error:
    print("[ERROR] one_to_one:", error)

pd.merge(clientes, ventas_2, on="customer_id", validate="one_to_many")  # [OK] un cliente → muchas ventas: es lo esperado
print("[OK] one_to_many: la relación es la esperada")

Filas de clientes: 4 · filas del join: 4
[ERROR] one_to_one: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  customer_id
           1 ...
[OK] one_to_many: la relación es la esperada


---
## 3. Transformación de datos

### ¿Por qué hay que transformar?

Al final, todo modelo de Machine Learning es una función $f(\mathbf{x}; \mathbf{w})$, donde $\mathbf{x}$ son las variables de entrada y $\mathbf{w}$ los parámetros que se aprenden en el entrenamiento.
Por ejemplo, una regresión lineal hace $\hat{y} = w_0 + w_1 x_1$, que es una **multiplicación de matrices**:

In [15]:
w = np.array([1, 2])  # w0 = 1 (intercepto), w1 = 2
X = np.array([[1.0, 0.5], [1.0, 0.0], [1.0, 1.0], [1.0, 5.0], [1.0, 8.0]])
y = X @ w  # equivale a np.matmul(X, w)
y

array([ 2.,  1.,  3., 11., 17.])

De aquí salen **dos consecuencias**:
1. **Todo tiene que ser número.** Un modelo no puede multiplicar `"Bogotá" × w`: hay que **codificar** las categóricas.
2. **La escala importa.** Si una variable va de 0 a 1 y otra de 0 a 1.000.000, la segunda domina en los modelos basados en distancias (KNN, SVM, K-means) y en los que usan descenso de gradiente. Hay que **escalar**.

### 3.1 Escalado

In [16]:
datos = pd.DataFrame({"categoria": ["A", "B", "A", "C", "B"], "edad": [20, 30, 20, 40, 30]})

# Estandarización (z-score): media 0, desviación 1   →  z = (x - μ) / σ
mu, sigma = datos["edad"].mean(), datos["edad"].std(ddof=0)
datos["edad_estandar_manual"] = (datos["edad"] - mu) / sigma
datos["edad_estandar_sklearn"] = StandardScaler().fit_transform(datos[["edad"]]).ravel()

# Normalización min-max: queda entre 0 y 1   →  x' = (x - min) / (max - min)
datos["edad_minmax"] = MinMaxScaler().fit_transform(datos[["edad"]]).ravel()
datos

,categoria,edad,edad_estandar_manual,edad_estandar_sklearn,edad_minmax
0,A,20,-1.069045,-1.069045,0.0
1,B,30,0.267261,0.267261,0.5
2,A,20,-1.069045,-1.069045,0.0
3,C,40,1.603567,1.603567,1.0
4,B,30,0.267261,0.267261,0.5


| | `StandardScaler` | `MinMaxScaler` |
|---|---|---|
| Resultado | Media 0, desviación 1 | Entre 0 y 1 |
| Con *outliers* | Más robusto | Un *outlier* "aplasta" al resto |
| Útil para | Regresión lineal/logística, SVM, PCA | Redes neuronales, imágenes |

> Los **árboles** (Decision Tree, Random Forest, XGBoost) **no necesitan** escalado: solo comparan si un valor es mayor o menor que un umbral.

### 3.2 Codificación de categóricas

| Método | Qué hace | Cuándo |
|---|---|---|
| **Label / Ordinal encoding** | A→0, B→1, C→2 | Solo si hay **orden real** (Bajo < Medio < Alto) o para árboles |
| **One-hot encoding** | Una columna 0/1 por categoría | Categorías **sin orden** (ciudad, color) |

In [17]:
# LabelEncoder: asigna números en orden alfabético (está pensado para el TARGET)
datos["categoria_label"] = LabelEncoder().fit_transform(datos["categoria"])

# One-hot
onehot = OneHotEncoder(sparse_output=False)
columnas_onehot = pd.DataFrame(
    onehot.fit_transform(datos[["categoria"]]), columns=onehot.get_feature_names_out(["categoria"])
)
pd.concat([datos[["categoria", "categoria_label"]], columnas_onehot], axis=1)

,categoria,categoria_label,categoria_A,categoria_B,categoria_C
0,A,0,1.0,0.0,0.0
1,B,1,0.0,1.0,0.0
2,A,0,1.0,0.0,0.0
3,C,2,0.0,0.0,1.0
4,B,1,0.0,1.0,0.0


**Cuidado con el label encoding en categorías sin orden:** si `Bogotá=0`, `Cali=1` y `Medellín=2`, una regresión lineal "cree" que Medellín es "el doble" que Cali.

Cuando **sí** hay orden, usa `OrdinalEncoder` y **dile tú el orden** (si no, usa el alfabético):

In [18]:
calidad = pd.DataFrame({"calidad": ["Regular", "Excelente", "Bueno", "Regular", "Bueno"]})
ordinal = OrdinalEncoder(categories=[["Regular", "Bueno", "Excelente"]])
calidad["calidad_ordinal"] = ordinal.fit_transform(calidad[["calidad"]]).ravel()
calidad

,calidad,calidad_ordinal
0,Regular,0.0
1,Excelente,2.0
2,Bueno,1.0
3,Regular,0.0
4,Bueno,1.0


### 3.3 Una advertencia que vale oro: *data leakage*

Los escaladores e imputadores **aprenden** algo de los datos (media, mínimo, categorías…).
Si los ajustas (`fit`) con **todos** los datos, incluido el test, el modelo "ve" información del futuro y las métricas salen infladas.

```python
# MAL
scaler.fit(X)                    # incluye test
# BIEN
scaler.fit(X_train)              # aprende solo del entrenamiento
X_test_esc = scaler.transform(X_test)
```

En el **módulo 05** los `Pipeline` resolverán esto automáticamente.

---
## Ejercicio guiado: viajes (20 min)

Una agencia de viajes nos pasa **dos tablas**: viajeros y viajes. Queremos responder **¿cuánto cuesta en promedio un día de viaje según el destino?**

In [19]:
viajeros = pd.DataFrame(
    {
        "user_id": [136, 284, 101, 529, 800, 823],
        "age": [None, 38, 30, 43, 49, 28],
        "name": ["Ana", "Jose", "Tomas", "Viviana", "Carolina", "Katerine"],
    }
)
viajes = pd.DataFrame(
    {
        "user_id": [101, 284, 136, 800, 101, 800, 823, 529, 284],
        "date_of_journey": [
            "2018-01-16", "2017-07-13", "2019-10-10", "2018/03/20", "2019-12-24",
            "2017-10-17", "2016/11/02", "2019/09/14", "2019-08-07",
        ],
        "duration": [3, 3, 2, 4, 2, 3.5, 4.5, 2.5, 4.3],
        "destination": [
            "Costa Rica", "colombia", "Colombia", "Costa_Rica", "Colombia/",
            "Colombia", "Costa Rica", "Colombia", "Costa_Rica",
        ],
        "cost": [None, 775, 587, 913, 1333, 825, 1046, 613, 970],
        "currency": [None, "EUR", "USD", "USD", "USD", "EUR", "USD", "USD", "USD"],
    }
)
display(viajeros, viajes)

,user_id,age,name
0,136,NaN,Ana
1,284,38.0,Jose
2,101,30.0,Tomas
3,529,43.0,Viviana
4,800,49.0,Carolina
5,823,28.0,Katerine


,user_id,date_of_journey,duration,destination,cost,currency
0,101,2018-01-16,3.0,Costa Rica,NaN,NaN
1,284,2017-07-13,3.0,colombia,775.0,EUR
2,136,2019-10-10,2.0,Colombia,587.0,USD
3,800,2018/03/20,4.0,Costa_Rica,913.0,USD
4,101,2019-12-24,2.0,Colombia/,1333.0,USD
5,800,2017-10-17,3.5,Colombia,825.0,EUR
6,823,2016/11/02,4.5,Costa Rica,1046.0,USD
7,529,2019/09/14,2.5,Colombia,613.0,USD
8,284,2019-08-07,4.3,Costa_Rica,970.0,USD


**Tareas** (en orden):

1. **Integrar:** une `viajes` con `viajeros` por `user_id`. ¿Qué tipo de join usarías? ¿Por qué?
2. **Fechas:** `date_of_journey` mezcla `-` y `/`. Conviértela a `datetime`.
3. **Moneda:** hay costos en EUR. Conviértelos a USD con la tasa `1.16 USD/EUR` y deja toda la columna `currency` en `"USD"`.
4. **Destino:** normaliza `"colombia"`, `"Colombia/"`, `"Costa_Rica"`… a solo **dos** valores: `"Colombia"` y `"Costa Rica"`.
5. **Nulos:** el costo faltante es de un viaje a Costa Rica. Imputa con la **media de Costa Rica** (no con la media global, ¿por qué?). La edad faltante, con la mediana.
6. **Nueva variable:** crea `costo_por_dia = cost / duration`.
7. **Responde:** ¿cuál destino es más caro por día?
8. **Transformar:** escala `cost` con `MinMaxScaler` y codifica `destination` (¿label o one-hot? ¿por qué?).
9. Leer documentación de guía para procesamiento de datos categoricos y númericos https://scikit-learn.org/stable/modules/preprocessing.html. 

Solución deberá ir en el repositorio del proyecto (haz un flork) y resolver en este mismo notebook. 

# Tu código aquí